# 03 — Align Candidate Point Events with NF3 NetFlow (`point_event_nf3_alignment_v1`)

This notebook aligns the frozen candidate low-altitude **point-event** table with NF3 NetFlow telemetry while preserving one output row per physical event.

It performs the following operations:

- Loads `candidate_point_events_v1_with_episode_recurrence.csv` from Notebook 02.
- Loads the narrowed NF3 / NF-UNSW-NB15-v3 table.
- Uses the aircraft-reported UTC timestamp as the physical event-time anchor.
- Uses interval-overlap logic: a flow matches when `flow_start <= window_end` and `flow_end >= window_start`.
- Creates primary ±120-second windows and ±300/±600-second sensitivity windows.
- Separates non-label cyber features from NF3 label-derived evaluation targets to prevent target leakage.
- Writes a primary fused feature bank, a long sensitivity table, target-only outcomes, and event-to-flow edges.

## Interpretation boundary

The FR24 and NF3 datasets are independent and are not site-coincident observations. When their UTC periods do not overlap, this notebook uses a deterministic controlled cyclic replay mapping. Any resulting association is a **modeled cross-domain temporal pairing**, not evidence that an aircraft event caused or accompanied a real network attack.

**Implementation correction:** this revision forces the NF3 interval index to nanoseconds and validates indexed overlap results against exact interval joins before producing artifacts.

In [ ]:
# Imports and scalar-validation helpers
from __future__ import annotations

from datetime import datetime
from pathlib import Path
from typing import Any, Iterable
import hashlib
import json
import math
import re
import warnings

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


def require_text(value: object, field_name: str) -> str:
    """Convert a required pandas/Python scalar to a clean string."""
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{field_name} cannot be missing.")

    if isinstance(value, bytes):
        text = value.decode("utf-8", errors="strict").strip()
    else:
        text = str(value).strip()

    if not text or text.lower() in {"nan", "nat", "none", "<na>"}:
        raise ValueError(f"{field_name} cannot be empty or missing.")
    return text


def require_int(value: object, field_name: str) -> int:
    """Convert a required pandas/Python scalar to a Python int."""
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{field_name} cannot be missing.")
    if isinstance(value, bool):
        return 1 if value else 0

    text = require_text(value, field_name)
    if re.fullmatch(r"[+-]?\d+", text):
        return int(text)
    if re.fullmatch(r"[+-]?\d+\.0+", text):
        return int(float(text))
    raise ValueError(f"{field_name} must be a whole number; received {value!r}.")


def require_float(value: object, field_name: str) -> float:
    """Convert a required pandas/Python scalar to a finite Python float."""
    text = require_text(value, field_name)
    try:
        number = float(text)
    except ValueError as exc:
        raise ValueError(f"{field_name} must be numeric; received {value!r}.") from exc
    if not math.isfinite(number):
        raise ValueError(f"{field_name} must be finite; received {value!r}.")
    return number


def require_utc_timestamp(value: object, field_name: str) -> pd.Timestamp:
    """Convert a required scalar to a timezone-aware UTC pandas Timestamp."""
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{field_name} cannot be missing.")

    if isinstance(value, pd.Timestamp):
        timestamp = value
    elif isinstance(value, datetime):
        timestamp = pd.Timestamp(value)
    elif isinstance(value, np.datetime64):
        timestamp = pd.Timestamp(value)
    else:
        timestamp = pd.Timestamp(require_text(value, field_name))

    if pd.isna(timestamp):
        raise ValueError(f"{field_name} cannot be missing.")
    if timestamp.tzinfo is None:
        timestamp = timestamp.tz_localize("UTC")
    else:
        timestamp = timestamp.tz_convert("UTC")
    return timestamp


In [ ]:
# Versioned configuration and project paths

ALIGNMENT_CONFIG = {
    "definition_name": "candidate_point_event_nf3_alignment",
    "alignment_version": "point_event_nf3_alignment_v1",
    "status": "frozen",
    "implementation_revision": "ns_safe_interval_index_2026_07_27",
    "physical_event_time_field": "aircraft_event_time_utc",
    "alignment_mode": "auto",  # auto, direct_utc, or controlled_cyclic_replay
    "direct_utc_minimum_coverage_fraction": 0.95,
    "windows": {
        "primary_120s": 120,
        "sensitivity_300s": 300,
        "sensitivity_600s": 600,
    },
    "primary_window": "primary_120s",
    "benign_label_values": ["BENIGN", "NORMAL", "0", "FALSE"],
    "short_flow_duration_threshold_s": 3600,
    "write_primary_edge_table": True,
    "edge_flush_every_events": 250,
    "compute_sha256": True,
}

EXPECTED_POINT_EVENT_ROWS = 12_185


def locate_project_root(start: Path) -> Path:
    """Find the nearest parent containing the Notebook 02 output."""
    relative_input = Path(
        "data/processed/point_event_v1/context_v1/"
        "candidate_point_events_v1_with_episode_recurrence.csv"
    )
    candidates = [start.resolve(), *start.resolve().parents]
    for candidate in candidates:
        if (candidate / relative_input).exists():
            return candidate
    return start.resolve()


PROJECT_ROOT = locate_project_root(Path.cwd())
CONTEXT_INPUT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "point_event_v1"
    / "context_v1"
    / "candidate_point_events_v1_with_episode_recurrence.csv"
)
NETFLOW_ROOT = PROJECT_ROOT / "data" / "netflow"
PREFERRED_NETFLOW_FILE = NETFLOW_ROOT / "nf_unsw_nb15_v3_narrowed.csv"
OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "point_event_v1"
    / "nf3_alignment_v1"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_FEATURES_OUT = OUTPUT_DIR / "event_nf3_window_features_v1.csv"
TARGETS_OUT = OUTPUT_DIR / "event_nf3_targets_v1.csv"
PRIMARY_FUSED_OUT = OUTPUT_DIR / "candidate_point_events_v1_with_context_and_nf3_primary.csv"
EDGES_OUT = OUTPUT_DIR / "event_flow_edges_primary_v1.csv"
FLOW_LABELS_OUT = OUTPUT_DIR / "nf3_flow_labels_v1.csv"
SUMMARY_OUT = OUTPUT_DIR / "point_event_nf3_alignment_v1_build_summary.json"
DICTIONARY_OUT = OUTPUT_DIR / "point_event_nf3_alignment_v1_data_dictionary.csv"
CHECKSUMS_OUT = OUTPUT_DIR / "SHA256SUMS.txt"

print("Project root:", PROJECT_ROOT)
print("Context input:", CONTEXT_INPUT)
print("NetFlow root:", NETFLOW_ROOT)
print("Output directory:", OUTPUT_DIR)


## 1. Load and validate the frozen physical/context table

Notebook 03 must preserve all 12,185 point-event rows. Episode and recurrence fields remain context; they are not collapsed into incident rows.

In [ ]:
if not CONTEXT_INPUT.exists():
    raise FileNotFoundError(
        "Notebook 02 output was not found. Expected:\n"
        f"  {CONTEXT_INPUT}\n"
        "Run 02_add_episode_and_recurrence_features.ipynb first."
    )

events = pd.read_csv(CONTEXT_INPUT, low_memory=False)
required_event_columns = {
    "event_id",
    "event_definition_version",
    "context_definition_version",
    "aircraft_event_time_utc",
    "aircraft_key",
}
missing_event_columns = required_event_columns - set(events.columns)
if missing_event_columns:
    raise ValueError(f"Physical/context input is missing columns: {sorted(missing_event_columns)}")

events["aircraft_event_time_utc"] = pd.to_datetime(
    events["aircraft_event_time_utc"], utc=True, errors="coerce"
)
if events["aircraft_event_time_utc"].isna().any():
    raise ValueError("The physical/context table contains invalid aircraft_event_time_utc values.")

assert events["event_id"].is_unique, "event_id must be unique before alignment."
assert len(events) == EXPECTED_POINT_EVENT_ROWS, (
    f"Expected {EXPECTED_POINT_EVENT_ROWS:,} events from Notebook 02, "
    f"but found {len(events):,}. Investigate before continuing."
)

events = events.sort_values(["aircraft_event_time_utc", "event_id"]).reset_index(drop=True)

print(f"Loaded {len(events):,} candidate point events with {len(events.columns)} columns.")
print("Physical event-time range:")
print("  first:", events["aircraft_event_time_utc"].min())
print("  last :", events["aircraft_event_time_utc"].max())
display(events[["event_id", "aircraft_key", "aircraft_event_time_utc", "episode_id", "prior_event_count"]].head())

## 2. Locate and inspect the NF3 source

The preferred location is:

```text
data/netflow/nf_unsw_nb15_v3_narrowed.csv
```

CSV and Parquet files are supported. If the preferred file is absent, the notebook uses the only supported file in `data/netflow`; if multiple candidates exist, set `PREFERRED_NETFLOW_FILE` explicitly.

In [ ]:
SUPPORTED_NETFLOW_SUFFIXES = {".csv", ".parquet", ".pq"}


def discover_netflow_file() -> Path:
    if PREFERRED_NETFLOW_FILE.exists():
        return PREFERRED_NETFLOW_FILE

    if not NETFLOW_ROOT.exists():
        raise FileNotFoundError(
            f"NetFlow directory does not exist: {NETFLOW_ROOT}\n"
            "Create data/netflow and place the narrowed NF3 file there."
        )

    candidates = sorted(
        path
        for path in NETFLOW_ROOT.iterdir()
        if path.is_file() and path.suffix.lower() in SUPPORTED_NETFLOW_SUFFIXES
    )
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            "No NF3 CSV or Parquet file was found in data/netflow. "
            "Expected nf_unsw_nb15_v3_narrowed.csv or one unambiguous supported file."
        )
    raise RuntimeError(
        "Multiple NetFlow files were found. Set PREFERRED_NETFLOW_FILE to the intended file:\n"
        + "\n".join(f"  - {path}" for path in candidates)
    )


NETFLOW_FILE = discover_netflow_file()
print("Using NF3 source:", NETFLOW_FILE)

In [ ]:
# Column-name normalization and schema resolution

def normalize_column_name(name: str) -> str:
    return re.sub(r"[^A-Z0-9]+", "_", str(name).upper()).strip("_")


def read_header(path: Path) -> list[str]:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return list(pd.read_csv(path, nrows=0).columns)
    if suffix in {".parquet", ".pq"}:
        return list(pd.read_parquet(path).columns)
    raise ValueError(f"Unsupported NetFlow file type: {suffix}")


raw_net_columns = read_header(NETFLOW_FILE)
normalized_to_actual: dict[str, str] = {}
for actual in raw_net_columns:
    normalized_to_actual.setdefault(normalize_column_name(actual), actual)


def resolve_first(candidates: Iterable[str]) -> str | None:
    for candidate in candidates:
        actual = normalized_to_actual.get(normalize_column_name(candidate))
        if actual is not None:
            return actual
    return None


def resolve_all(candidates: Iterable[str]) -> list[str]:
    found: list[str] = []
    for candidate in candidates:
        actual = normalized_to_actual.get(normalize_column_name(candidate))
        if actual is not None and actual not in found:
            found.append(actual)
    return found


COLUMN_CANDIDATES = {
    "start_ms": [
        "FLOW_START_MILLISECONDS", "FLOW_START_MS", "START_MILLISECONDS",
        "START_MS", "FLOW_START"
    ],
    "end_ms": [
        "FLOW_END_MILLISECONDS", "FLOW_END_MS", "END_MILLISECONDS",
        "END_MS", "FLOW_END"
    ],
    "binary_label": ["LABEL", "BINARY_LABEL", "IS_ATTACK"],
    "attack_category": [
        "ATTACK", "ATTACK_CATEGORY", "ATTACK_TYPE", "CATEGORY"
    ],
    "protocol": ["PROTOCOL", "L4_PROTO", "PROTOCOL_ID", "PROTOCOL_IDENTIFIER"],
    "src_ip": ["IPV4_SRC_ADDR", "IPV6_SRC_ADDR", "SRC_IP", "SOURCE_IP"],
    "dst_ip": ["IPV4_DST_ADDR", "IPV6_DST_ADDR", "DST_IP", "DESTINATION_IP"],
    "src_port": ["L4_SRC_PORT", "SRC_PORT", "SOURCE_PORT"],
    "dst_port": ["L4_DST_PORT", "DST_PORT", "DESTINATION_PORT"],
}

resolved = {role: resolve_first(candidates) for role, candidates in COLUMN_CANDIDATES.items()}
if resolved["start_ms"] is None or resolved["end_ms"] is None:
    raise ValueError(
        "NF3 must contain flow start and end millisecond columns. "
        f"Available columns: {raw_net_columns}"
    )
if resolved["binary_label"] is None:
    raise ValueError("NF3 binary Label column was not found.")
if resolved["attack_category"] is None:
    raise ValueError("NF3 Attack category column was not found.")


# Prefer a single total measure; otherwise sum directional measures.
def resolve_measure_columns(
    total_candidates: list[str],
    directional_candidates: list[str],
) -> list[str]:
    total = resolve_first(total_candidates)
    if total is not None:
        return [total]
    return resolve_all(directional_candidates)


packet_columns = resolve_measure_columns(
    ["TOTAL_PACKETS", "TOTAL_PKTS", "BIDIRECTIONAL_PACKETS", "PACKETS"],
    [
        "IN_PKTS", "OUT_PKTS", "SRC_TO_DST_PACKETS", "DST_TO_SRC_PACKETS",
        "PACKETS_IN", "PACKETS_OUT"
    ],
)
byte_columns = resolve_measure_columns(
    ["TOTAL_BYTES", "BIDIRECTIONAL_BYTES", "BYTES"],
    [
        "IN_BYTES", "OUT_BYTES", "SRC_TO_DST_BYTES", "DST_TO_SRC_BYTES",
        "BYTES_IN", "BYTES_OUT"
    ],
)
iat_columns = [
    column for column in raw_net_columns
    if "IAT" in normalize_column_name(column)
]

schema_report = pd.DataFrame(
    [
        {"role": "flow_start_ms", "resolved_column": resolved["start_ms"]},
        {"role": "flow_end_ms", "resolved_column": resolved["end_ms"]},
        {
            "role": "binary_label_target_only",
            "resolved_column": resolved["binary_label"],
        },
        {
            "role": "attack_category_target_only",
            "resolved_column": resolved["attack_category"],
        },
        {"role": "protocol", "resolved_column": resolved["protocol"]},
        {"role": "source_ip", "resolved_column": resolved["src_ip"]},
        {"role": "destination_ip", "resolved_column": resolved["dst_ip"]},
        {"role": "source_port", "resolved_column": resolved["src_port"]},
        {"role": "destination_port", "resolved_column": resolved["dst_port"]},
        {
            "role": "packet_measure_columns",
            "resolved_column": ", ".join(packet_columns) or None,
        },
        {
            "role": "byte_measure_columns",
            "resolved_column": ", ".join(byte_columns) or None,
        },
        {
            "role": "IAT_columns",
            "resolved_column": ", ".join(iat_columns) or None,
        },
    ]
)
display(schema_report)


## 3. Load and normalize NF3 event times

NF3 labels are retained only for the separate target table. They are not inserted into the model-feature bank.

In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


start_column = require_text(resolved["start_ms"], "resolved start_ms column")
end_column = require_text(resolved["end_ms"], "resolved end_ms column")
binary_label_column = require_text(
    resolved["binary_label"],
    "resolved binary_label column",
)
attack_category_column = require_text(
    resolved["attack_category"],
    "resolved attack_category column",
)

selected_columns = {
    start_column,
    end_column,
    binary_label_column,
    attack_category_column,
    *packet_columns,
    *byte_columns,
    *iat_columns,
}
for role in ["protocol", "src_ip", "dst_ip", "src_port", "dst_port"]:
    source_column = resolved[role]
    if source_column is not None:
        selected_columns.add(source_column)
selected_columns = [column for column in raw_net_columns if column in selected_columns]

if NETFLOW_FILE.suffix.lower() == ".csv":
    net_raw = pd.read_csv(NETFLOW_FILE, usecols=selected_columns, low_memory=False)
else:
    net_raw = pd.read_parquet(NETFLOW_FILE, columns=selected_columns)

source_netflow_rows = len(net_raw)
net_raw["source_row_number"] = np.arange(source_netflow_rows, dtype=np.int64)

start_ms = pd.to_numeric(net_raw[start_column], errors="coerce")
end_ms = pd.to_numeric(net_raw[end_column], errors="coerce")
net_raw["flow_start_utc"] = pd.to_datetime(start_ms, unit="ms", utc=True, errors="coerce")
net_raw["flow_end_utc"] = pd.to_datetime(end_ms, unit="ms", utc=True, errors="coerce")

valid_time_mask = (
    net_raw["flow_start_utc"].notna()
    & net_raw["flow_end_utc"].notna()
    & (net_raw["flow_end_utc"] >= net_raw["flow_start_utc"])
)
invalid_netflow_time_rows = require_int(
    (~valid_time_mask).sum(),
    "invalid_netflow_time_rows",
)
net = net_raw.loc[valid_time_mask].copy()
net["flow_duration_s"] = (
    net["flow_end_utc"] - net["flow_start_utc"]
).dt.total_seconds()


# Create combined packet/byte measures without using target columns.
def numeric_row_sum(frame: pd.DataFrame, columns: list[str]) -> pd.Series:
    if not columns:
        return pd.Series(np.nan, index=frame.index, dtype="float64")
    numeric = frame[columns].apply(pd.to_numeric, errors="coerce")
    return numeric.sum(axis=1, min_count=1)


net["packet_total"] = numeric_row_sum(net, packet_columns)
net["byte_total"] = numeric_row_sum(net, byte_columns)

for column in iat_columns:
    net[column] = pd.to_numeric(net[column], errors="coerce")

net = net.sort_values(
    ["flow_start_utc", "flow_end_utc", "source_row_number"]
).reset_index(drop=True)
net["flow_id"] = net["source_row_number"].map(
    lambda value: f"NF3-{require_int(value, 'source_row_number'):012d}"
)

# Preserve both source labels in a target-only flow lookup.
flow_labels = net[
    [
        "flow_id",
        "source_row_number",
        binary_label_column,
        attack_category_column,
    ]
].copy()
flow_labels = flow_labels.rename(
    columns={
        binary_label_column: "nf3_binary_label_raw",
        attack_category_column: "nf3_attack_category_raw",
    }
)
flow_labels["nf3_binary_label"] = pd.to_numeric(
    flow_labels["nf3_binary_label_raw"],
    errors="coerce",
).astype("Int64")
flow_labels["nf3_attack_category"] = (
    flow_labels["nf3_attack_category_raw"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)
assert flow_labels["flow_id"].is_unique

nf3_start = require_utc_timestamp(net["flow_start_utc"].min(), "nf3_start")
nf3_end = require_utc_timestamp(net["flow_end_utc"].max(), "nf3_end")
nf3_span_s = require_float(
    (nf3_end - nf3_start).total_seconds(),
    "nf3_span_s",
)

maximum_window_s = max(
    require_int(value, f"window {name}")
    for name, value in ALIGNMENT_CONFIG["windows"].items()
)
if nf3_span_s <= 2 * maximum_window_s:
    raise ValueError(
        "The usable NF3 time span is too short for the configured alignment windows. "
        f"Span={nf3_span_s} seconds."
    )

print(f"Loaded {source_netflow_rows:,} NF3 rows; retained {len(net):,} with valid intervals.")
print(f"Invalid or reversed flow-time rows: {invalid_netflow_time_rows:,}")
print("NF3 time range:")
print("  start:", nf3_start)
print("  end  :", nf3_end)
print(f"  span : {nf3_span_s:,.1f} seconds")
print("Binary Label column:", binary_label_column)
print("Attack category column:", attack_category_column)


## 4. Select the temporal mapping

- `direct_utc`: uses the original aircraft event timestamp. Appropriate only for genuinely overlapping, site-coincident timelines.
- `controlled_cyclic_replay`: maps elapsed physical time into the NF3 time span using modulo arithmetic. It preserves event order and short-term gaps within each replay cycle but is explicitly synthetic.
- `auto`: selects direct UTC only when at least 95% of point events fall inside the NF3 time span; otherwise it selects controlled cyclic replay.

In [ ]:
physical_time = events[ALIGNMENT_CONFIG["physical_event_time_field"]]
direct_coverage_fraction = float(physical_time.between(nf3_start, nf3_end, inclusive="both").mean())
requested_mode = ALIGNMENT_CONFIG["alignment_mode"]

if requested_mode == "auto":
    alignment_mode = (
        "direct_utc"
        if direct_coverage_fraction >= ALIGNMENT_CONFIG["direct_utc_minimum_coverage_fraction"]
        else "controlled_cyclic_replay"
    )
elif requested_mode in {"direct_utc", "controlled_cyclic_replay"}:
    alignment_mode = requested_mode
else:
    raise ValueError(f"Unsupported alignment mode: {requested_mode}")

mapped = events[["event_id", ALIGNMENT_CONFIG["physical_event_time_field"]]].copy()
mapped = mapped.rename(columns={ALIGNMENT_CONFIG["physical_event_time_field"]: "physical_event_time_utc"})

if alignment_mode == "direct_utc":
    mapped["aligned_event_time_utc"] = mapped["physical_event_time_utc"]
    mapped["replay_cycle_index"] = 0
    mapped["alignment_is_controlled"] = 0
else:
    physical_t0 = mapped["physical_event_time_utc"].min()
    elapsed_s = (mapped["physical_event_time_utc"] - physical_t0).dt.total_seconds()
    mapped["replay_cycle_index"] = np.floor(elapsed_s / nf3_span_s).astype("int64")
    replay_offset_s = np.mod(elapsed_s, nf3_span_s)
    mapped["aligned_event_time_utc"] = nf3_start + pd.to_timedelta(replay_offset_s, unit="s")
    mapped["alignment_is_controlled"] = 1

mapped["alignment_mode"] = alignment_mode
mapped["alignment_version"] = ALIGNMENT_CONFIG["alignment_version"]
mapped["alignment_time_shift_s"] = (
    mapped["aligned_event_time_utc"] - mapped["physical_event_time_utc"]
).dt.total_seconds()

print("Requested mode:", requested_mode)
print("Selected mode :", alignment_mode)
print(f"Direct UTC coverage fraction: {direct_coverage_fraction:.4f}")
if alignment_mode != "direct_utc":
    warnings.warn(
        "Controlled cyclic replay is active. The resulting physical/cyber associations "
        "are modeled pairings and must not be interpreted as causal or site-coincident observations."
    )

display(mapped.head())

## 5. Build interval-overlap features and primary event-to-flow edges

For each event and each configured half-window `m`, the window is:

```text
[aligned_event_time - m, aligned_event_time + m]
```

A flow is matched when:

```text
flow_start_utc <= window_end_utc
and
flow_end_utc >= window_start_utc
```

Windows that reach beyond the NF3 replay bounds are clipped and flagged.

In [ ]:
# Efficient exact overlap lookup.
# Short flows are indexed by start time; long flows are checked separately.
SHORT_FLOW_THRESHOLD_S = float(ALIGNMENT_CONFIG["short_flow_duration_threshold_s"])
short_flows = net.loc[net["flow_duration_s"] <= SHORT_FLOW_THRESHOLD_S].copy()
long_flows = net.loc[net["flow_duration_s"] > SHORT_FLOW_THRESHOLD_S].copy()
short_flows = short_flows.sort_values("flow_start_utc").reset_index(drop=True)
# Force nanosecond resolution explicitly. Some pandas versions preserve the
# source millisecond unit when casting timezone-aware datetimes to int64,
# while Timestamp.value is always nanoseconds. Mixing those units silently
# makes every searchsorted lookup miss the NF3 timeline.
short_start_ns = (
    short_flows["flow_start_utc"]
    .to_numpy(dtype="datetime64[ns]")
    .astype(np.int64, copy=False)
)
if short_start_ns.size:
    first_expected_ns = require_utc_timestamp(
        short_flows.iloc[0]["flow_start_utc"],
        "first short-flow start",
    ).value
    last_expected_ns = require_utc_timestamp(
        short_flows.iloc[-1]["flow_start_utc"],
        "last short-flow start",
    ).value
    if (
        require_int(short_start_ns[0], "first indexed flow-start ns") != first_expected_ns
        or require_int(short_start_ns[-1], "last indexed flow-start ns") != last_expected_ns
    ):
        raise RuntimeError(
            "NF3 interval index is not in nanoseconds; overlap lookup would be invalid."
        )

print(f"Short flows (<= {SHORT_FLOW_THRESHOLD_S:g}s): {len(short_flows):,}")
print(f"Long flows  (>  {SHORT_FLOW_THRESHOLD_S:g}s): {len(long_flows):,}")
if len(long_flows) > max(10_000, 0.05 * len(net)):
    warnings.warn(
        "Many flows exceed the short-flow threshold. Alignment remains exact but may run slowly."
    )


def find_overlapping_flows(window_start: pd.Timestamp, window_end: pd.Timestamp) -> pd.DataFrame:
    if pd.isna(window_start) or pd.isna(window_end) or window_start > window_end:
        return net.iloc[0:0]

    lower_start = window_start - pd.Timedelta(seconds=SHORT_FLOW_THRESHOLD_S)
    left = require_int(
        np.searchsorted(short_start_ns, lower_start.value, side="left"),
        "short-flow search left index",
    )
    right = require_int(
        np.searchsorted(short_start_ns, window_end.value, side="right"),
        "short-flow search right index",
    )
    short_candidates = short_flows.iloc[left:right]
    short_matches = short_candidates.loc[
        (short_candidates["flow_start_utc"] <= window_end)
        & (short_candidates["flow_end_utc"] >= window_start)
    ]

    if long_flows.empty:
        return short_matches

    long_matches = long_flows.loc[
        (long_flows["flow_start_utc"] <= window_end)
        & (long_flows["flow_end_utc"] >= window_start)
    ]
    if long_matches.empty:
        return short_matches
    if short_matches.empty:
        return long_matches
    return pd.concat([short_matches, long_matches], ignore_index=True)


def safe_stat(series: pd.Series, method: str) -> float:
    numeric = pd.to_numeric(series, errors="coerce").dropna()
    if numeric.empty:
        return np.nan
    if method == "sum":
        return float(numeric.sum())
    if method == "mean":
        return float(numeric.mean())
    if method == "median":
        return float(numeric.median())
    if method == "std":
        return float(numeric.std(ddof=0))
    if method == "max":
        return float(numeric.max())
    if method == "min":
        return float(numeric.min())
    raise ValueError(method)


BENIGN_VALUES = {str(value).strip().upper() for value in ALIGNMENT_CONFIG["benign_label_values"]}


def classify_relation(flow_start: pd.Timestamp, flow_end: pd.Timestamp, event_time: pd.Timestamp) -> str:
    if flow_end < event_time:
        return "pre_event"
    if flow_start > event_time:
        return "post_event"
    return "overlaps_event_time"


def aggregate_features(
    matched: pd.DataFrame,
    event_time: pd.Timestamp,
    window_start: pd.Timestamp,
    window_end: pd.Timestamp,
) -> dict[str, Any]:
    feature: dict[str, Any] = {
        "nf3_flow_count": int(len(matched)),
        "nf3_flow_duration_sum_s": 0.0 if matched.empty else safe_stat(matched["flow_duration_s"], "sum"),
        "nf3_flow_duration_mean_s": np.nan if matched.empty else safe_stat(matched["flow_duration_s"], "mean"),
        "nf3_flow_duration_median_s": np.nan if matched.empty else safe_stat(matched["flow_duration_s"], "median"),
        "nf3_flow_duration_std_s": np.nan if matched.empty else safe_stat(matched["flow_duration_s"], "std"),
        "nf3_flow_duration_max_s": np.nan if matched.empty else safe_stat(matched["flow_duration_s"], "max"),
        "nf3_packet_total_sum": 0.0 if matched.empty and packet_columns else np.nan,
        "nf3_packet_total_mean": np.nan,
        "nf3_packet_total_max": np.nan,
        "nf3_byte_total_sum": 0.0 if matched.empty and byte_columns else np.nan,
        "nf3_byte_total_mean": np.nan,
        "nf3_byte_total_max": np.nan,
        "nf3_protocol_unique_count": 0 if resolved["protocol"] is not None else np.nan,
        "nf3_src_ip_unique_count": 0 if resolved["src_ip"] is not None else np.nan,
        "nf3_dst_ip_unique_count": 0 if resolved["dst_ip"] is not None else np.nan,
        "nf3_src_port_unique_count": 0 if resolved["src_port"] is not None else np.nan,
        "nf3_dst_port_unique_count": 0 if resolved["dst_port"] is not None else np.nan,
        "nf3_pre_event_flow_count": 0,
        "nf3_at_event_flow_count": 0,
        "nf3_post_event_flow_count": 0,
        "nf3_nearest_flow_start_lag_s": np.nan,
        "nf3_first_flow_start_lag_s": np.nan,
        "nf3_last_flow_start_lag_s": np.nan,
        "nf3_total_overlap_duration_s": 0.0,
        "nf3_mean_overlap_duration_s": np.nan,
        "nf3_max_overlap_duration_s": np.nan,
    }

    for column in iat_columns:
        normalized = normalize_column_name(column).lower()
        feature[f"nf3_iat_{normalized}_mean"] = np.nan
        feature[f"nf3_iat_{normalized}_std"] = np.nan

    if matched.empty:
        return feature

    if packet_columns:
        feature["nf3_packet_total_sum"] = safe_stat(matched["packet_total"], "sum")
        feature["nf3_packet_total_mean"] = safe_stat(matched["packet_total"], "mean")
        feature["nf3_packet_total_max"] = safe_stat(matched["packet_total"], "max")
    if byte_columns:
        feature["nf3_byte_total_sum"] = safe_stat(matched["byte_total"], "sum")
        feature["nf3_byte_total_mean"] = safe_stat(matched["byte_total"], "mean")
        feature["nf3_byte_total_max"] = safe_stat(matched["byte_total"], "max")

    for role, output_name in [
        ("protocol", "nf3_protocol_unique_count"),
        ("src_ip", "nf3_src_ip_unique_count"),
        ("dst_ip", "nf3_dst_ip_unique_count"),
        ("src_port", "nf3_src_port_unique_count"),
        ("dst_port", "nf3_dst_port_unique_count"),
    ]:
        column = resolved[role]
        if column is not None:
            feature[output_name] = require_int(
                matched[column].dropna().nunique(),
                output_name,
            )

    relations = matched.apply(
        lambda row: classify_relation(row["flow_start_utc"], row["flow_end_utc"], event_time),
        axis=1,
    )
    relation_counts = relations.value_counts()
    feature["nf3_pre_event_flow_count"] = require_int(
        relation_counts.get("pre_event", 0),
        "nf3_pre_event_flow_count",
    )
    feature["nf3_at_event_flow_count"] = require_int(
        relation_counts.get("overlaps_event_time", 0),
        "nf3_at_event_flow_count",
    )
    feature["nf3_post_event_flow_count"] = require_int(
        relation_counts.get("post_event", 0),
        "nf3_post_event_flow_count",
    )

    lags = (matched["flow_start_utc"] - event_time).dt.total_seconds()
    feature["nf3_first_flow_start_lag_s"] = float(lags.min())
    feature["nf3_last_flow_start_lag_s"] = float(lags.max())
    feature["nf3_nearest_flow_start_lag_s"] = float(lags.loc[lags.abs().idxmin()])

    overlap_start = matched["flow_start_utc"].where(
        matched["flow_start_utc"] >= window_start, window_start
    )
    overlap_end = matched["flow_end_utc"].where(
        matched["flow_end_utc"] <= window_end, window_end
    )
    overlap_s = (overlap_end - overlap_start).dt.total_seconds().clip(lower=0)
    feature["nf3_total_overlap_duration_s"] = float(overlap_s.sum())
    feature["nf3_mean_overlap_duration_s"] = float(overlap_s.mean())
    feature["nf3_max_overlap_duration_s"] = float(overlap_s.max())

    for column in iat_columns:
        normalized = normalize_column_name(column).lower()
        feature[f"nf3_iat_{normalized}_mean"] = safe_stat(matched[column], "mean")
        feature[f"nf3_iat_{normalized}_std"] = safe_stat(matched[column], "std")

    return feature


def aggregate_targets(matched: pd.DataFrame) -> dict[str, Any]:
    empty_result: dict[str, Any] = {
        "nf3_binary_label_available_flag": 1,
        "nf3_attack_category_available_flag": 1,
        "cyber_attack_window_label": 0,
        "nf3_attack_flow_count": 0,
        "nf3_benign_flow_count": 0,
        "nf3_unknown_binary_label_count": 0,
        "nf3_attack_flow_fraction": 0.0,
        "nf3_distinct_attack_category_count": 0,
        "nf3_attack_categories_present": None,
        "nf3_attack_category_counts_json": None,
    }
    if matched.empty:
        return empty_result

    binary_raw = matched[binary_label_column]
    binary_numeric = pd.to_numeric(binary_raw, errors="coerce")
    binary_text = binary_raw.astype("string").str.strip().str.upper()

    known_numeric_mask = binary_numeric.notna()
    known_text_mask = binary_text.notna() & binary_text.ne("")
    known_binary_mask = known_numeric_mask | known_text_mask

    attack_mask = pd.Series(False, index=matched.index, dtype=bool)
    attack_mask.loc[known_numeric_mask] = (
        binary_numeric.loc[known_numeric_mask] != 0
    )
    text_only_mask = ~known_numeric_mask & known_text_mask
    attack_mask.loc[text_only_mask] = ~binary_text.loc[
        text_only_mask
    ].isin(BENIGN_VALUES)

    benign_mask = known_binary_mask & ~attack_mask
    unknown_mask = ~known_binary_mask

    attack_count = require_int(attack_mask.sum(), "nf3_attack_flow_count")
    benign_count = require_int(benign_mask.sum(), "nf3_benign_flow_count")
    unknown_count = require_int(
        unknown_mask.sum(),
        "nf3_unknown_binary_label_count",
    )
    total_count = len(matched)

    category_raw = matched[attack_category_column].astype("string").str.strip()
    category_normalized = category_raw.str.upper()
    valid_category_mask = (
        category_raw.notna()
        & category_raw.ne("")
        & ~category_normalized.isin(BENIGN_VALUES)
    )
    category_values = category_raw.loc[attack_mask & valid_category_mask]
    category_counts = category_values.value_counts().sort_index()
    attack_categories = category_counts.index.astype(str).tolist()
    attack_category_counts = {
        str(category): require_int(count, f"category count for {category}")
        for category, count in category_counts.items()
    }

    return {
        "nf3_binary_label_available_flag": 1,
        "nf3_attack_category_available_flag": 1,
        "cyber_attack_window_label": 1 if attack_count > 0 else 0,
        "nf3_attack_flow_count": attack_count,
        "nf3_benign_flow_count": benign_count,
        "nf3_unknown_binary_label_count": unknown_count,
        "nf3_attack_flow_fraction": (
            float(attack_count / total_count) if total_count else 0.0
        ),
        "nf3_distinct_attack_category_count": len(attack_categories),
        "nf3_attack_categories_present": (
            "|".join(attack_categories) if attack_categories else None
        ),
        "nf3_attack_category_counts_json": (
            json.dumps(attack_category_counts, sort_keys=True)
            if attack_category_counts
            else None
        ),
    }


def build_edge_rows(
    event_id: str,
    event_time: pd.Timestamp,
    window_name: str,
    window_start: pd.Timestamp,
    window_end: pd.Timestamp,
    matched: pd.DataFrame,
) -> pd.DataFrame:
    edge_columns = [
        "event_id", "flow_id", "window_name", "aligned_event_time_utc",
        "window_start_utc", "window_end_utc", "flow_start_utc", "flow_end_utc",
        "signed_flow_start_lag_s", "absolute_flow_start_lag_s", "overlap_seconds",
        "relation_type", "source_row_number", "protocol", "src_ip", "dst_ip",
        "src_port", "dst_port",
    ]
    if matched.empty:
        return pd.DataFrame(columns=edge_columns)

    edge = pd.DataFrame({
        "event_id": event_id,
        "flow_id": matched["flow_id"].values,
        "window_name": window_name,
        "aligned_event_time_utc": event_time,
        "window_start_utc": window_start,
        "window_end_utc": window_end,
        "flow_start_utc": matched["flow_start_utc"].values,
        "flow_end_utc": matched["flow_end_utc"].values,
        "source_row_number": matched["source_row_number"].values,
    })
    edge["signed_flow_start_lag_s"] = (
        pd.to_datetime(edge["flow_start_utc"], utc=True) - event_time
    ).dt.total_seconds()
    edge["absolute_flow_start_lag_s"] = edge["signed_flow_start_lag_s"].abs()
    overlap_start = pd.to_datetime(edge["flow_start_utc"], utc=True).where(
        pd.to_datetime(edge["flow_start_utc"], utc=True) >= window_start, window_start
    )
    overlap_end = pd.to_datetime(edge["flow_end_utc"], utc=True).where(
        pd.to_datetime(edge["flow_end_utc"], utc=True) <= window_end, window_end
    )
    edge["overlap_seconds"] = (overlap_end - overlap_start).dt.total_seconds().clip(lower=0)
    edge["relation_type"] = [
        classify_relation(start, end, event_time)
        for start, end in zip(
            pd.to_datetime(edge["flow_start_utc"], utc=True),
            pd.to_datetime(edge["flow_end_utc"], utc=True),
        )
    ]

    optional_roles = {
        "protocol": resolved["protocol"],
        "src_ip": resolved["src_ip"],
        "dst_ip": resolved["dst_ip"],
        "src_port": resolved["src_port"],
        "dst_port": resolved["dst_port"],
    }
    for output_column, source_column in optional_roles.items():
        edge[output_column] = matched[source_column].values if source_column is not None else np.nan
    return edge[edge_columns]


In [ ]:
# Validate the optimized interval lookup against an exact boolean-mask join.
# This detects timestamp-unit mismatches before the full 12,185-event alignment runs.
probe_positions = sorted({0, len(mapped) // 2, len(mapped) - 1})
probe_rows: list[dict[str, Any]] = []
probe_half_width_s = require_int(
    ALIGNMENT_CONFIG["windows"][ALIGNMENT_CONFIG["primary_window"]],
    "primary probe half-width",
)

for probe_position in probe_positions:
    probe_time = require_utc_timestamp(
        mapped.iloc[probe_position]["aligned_event_time_utc"],
        f"probe aligned time at position {probe_position}",
    )
    probe_start = max(
        probe_time - pd.Timedelta(seconds=probe_half_width_s),
        nf3_start,
    )
    probe_end = min(
        probe_time + pd.Timedelta(seconds=probe_half_width_s),
        nf3_end,
    )

    indexed_matches = find_overlapping_flows(probe_start, probe_end)
    exact_matches = net.loc[
        (net["flow_start_utc"] <= probe_end)
        & (net["flow_end_utc"] >= probe_start)
    ]

    indexed_ids = set(indexed_matches["flow_id"].astype(str))
    exact_ids = set(exact_matches["flow_id"].astype(str))
    match = indexed_ids == exact_ids
    probe_rows.append(
        {
            "probe_position": probe_position,
            "aligned_event_time_utc": probe_time,
            "indexed_overlap_count": len(indexed_ids),
            "exact_overlap_count": len(exact_ids),
            "exact_match": match,
        }
    )
    if not match:
        missing_from_index = sorted(exact_ids - indexed_ids)[:5]
        unexpected_in_index = sorted(indexed_ids - exact_ids)[:5]
        raise AssertionError(
            "Optimized NF3 overlap lookup disagrees with the exact interval join. "
            f"position={probe_position}, missing={missing_from_index}, "
            f"unexpected={unexpected_in_index}."
        )

probe_report = pd.DataFrame(probe_rows)
display(probe_report)
print("NF3 interval-index regression checks passed.")


In [ ]:
# Run the alignment for all configured windows.
window_records: list[dict[str, Any]] = []
target_records: list[dict[str, Any]] = []
primary_edge_count = 0
edge_header_written = False
if EDGES_OUT.exists():
    EDGES_OUT.unlink()
edge_buffer: list[pd.DataFrame] = []

mapped_lookup = mapped.set_index("event_id")
primary_window_name = require_text(
    ALIGNMENT_CONFIG["primary_window"],
    "primary_window",
)
edge_flush_every_events = require_int(
    ALIGNMENT_CONFIG["edge_flush_every_events"],
    "edge_flush_every_events",
)

for event_number, event in enumerate(events.itertuples(index=False), start=1):
    event_id = require_text(event.event_id, "event_id")
    map_row = mapped_lookup.loc[event_id]

    aligned_time = require_utc_timestamp(
        map_row["aligned_event_time_utc"],
        "aligned_event_time_utc",
    )
    alignment_is_controlled = require_int(
        map_row["alignment_is_controlled"],
        "alignment_is_controlled",
    )
    replay_cycle_index = require_int(
        map_row["replay_cycle_index"],
        "replay_cycle_index",
    )
    physical_event_time = require_utc_timestamp(
        map_row["physical_event_time_utc"],
        "physical_event_time_utc",
    )
    alignment_time_shift_s = require_float(
        map_row["alignment_time_shift_s"],
        "alignment_time_shift_s",
    )

    for window_name_raw, half_window_raw in ALIGNMENT_CONFIG["windows"].items():
        window_name = require_text(window_name_raw, "window_name")
        half_window_s = require_int(
            half_window_raw,
            f"window_half_width_s for {window_name}",
        )
        requested_start = aligned_time - pd.Timedelta(seconds=half_window_s)
        requested_end = aligned_time + pd.Timedelta(seconds=half_window_s)
        window_start = max(requested_start, nf3_start)
        window_end = min(requested_end, nf3_end)
        boundary_truncated = (
            1 if window_start != requested_start or window_end != requested_end else 0
        )

        matched = find_overlapping_flows(window_start, window_end)
        base_record = {
            "event_id": event_id,
            "alignment_version": ALIGNMENT_CONFIG["alignment_version"],
            "alignment_mode": alignment_mode,
            "alignment_is_controlled": alignment_is_controlled,
            "physical_event_time_utc": physical_event_time,
            "aligned_event_time_utc": aligned_time,
            "alignment_time_shift_s": alignment_time_shift_s,
            "replay_cycle_index": replay_cycle_index,
            "window_name": window_name,
            "window_half_width_s": half_window_s,
            "window_start_utc": window_start,
            "window_end_utc": window_end,
            "window_boundary_truncated_flag": boundary_truncated,
        }
        feature_record = dict(base_record)
        feature_record.update(
            aggregate_features(matched, aligned_time, window_start, window_end)
        )
        window_records.append(feature_record)

        target_record = {
            key: base_record[key]
            for key in [
                "event_id",
                "alignment_version",
                "alignment_mode",
                "alignment_is_controlled",
                "window_name",
                "window_half_width_s",
            ]
        }
        target_record.update(aggregate_targets(matched))
        target_records.append(target_record)

        if (
            ALIGNMENT_CONFIG["write_primary_edge_table"]
            and window_name == primary_window_name
            and not matched.empty
        ):
            edge = build_edge_rows(
                event_id,
                aligned_time,
                window_name,
                window_start,
                window_end,
                matched,
            )
            primary_edge_count += len(edge)
            edge_buffer.append(edge)

    if (
        ALIGNMENT_CONFIG["write_primary_edge_table"]
        and edge_buffer
        and event_number % edge_flush_every_events == 0
    ):
        edge_batch = pd.concat(edge_buffer, ignore_index=True)
        edge_batch.to_csv(
            EDGES_OUT,
            mode="a",
            header=not edge_header_written,
            index=False,
        )
        edge_header_written = True
        edge_buffer.clear()

    if event_number % 1000 == 0 or event_number == len(events):
        print(f"Aligned {event_number:,} / {len(events):,} point events...")

if edge_buffer:
    edge_batch = pd.concat(edge_buffer, ignore_index=True)
    edge_batch.to_csv(
        EDGES_OUT,
        mode="a",
        header=not edge_header_written,
        index=False,
    )
    edge_header_written = True
    edge_buffer.clear()

window_features = pd.DataFrame(window_records)
targets = pd.DataFrame(target_records)

# Ensure an edge file exists even when no primary-window flows match.
if ALIGNMENT_CONFIG["write_primary_edge_table"] and not EDGES_OUT.exists():
    build_edge_rows(
        "",
        pd.Timestamp(nf3_start),
        primary_window_name,
        nf3_start,
        nf3_start,
        net.iloc[0:0],
    ).to_csv(EDGES_OUT, index=False)

print("Window-feature rows:", f"{len(window_features):,}")
print("Target rows:", f"{len(targets):,}")
print("Primary event-flow edges:", f"{primary_edge_count:,}")

probe_expected_any_flow = bool(probe_report["exact_overlap_count"].gt(0).any())
if probe_expected_any_flow and not window_features["nf3_flow_count"].gt(0).any():
    raise RuntimeError(
        "Exact overlap probes found NF3 flows, but the completed alignment produced zero "
        "matched windows. Stop before modeling and inspect the interval-index implementation."
    )


## 6. Create the primary fused feature bank

Only the non-label NF3 aggregates from the primary ±120-second window are joined to the physical/context table. NF3 attack labels and attack counts remain in `event_nf3_targets_v1.csv`.

In [ ]:
primary_features = window_features.loc[
    window_features["window_name"].eq(primary_window_name)
].copy()

# Prevent duplicate physical timestamp columns when merging.
primary_features = primary_features.drop(
    columns=["physical_event_time_utc"],
    errors="ignore",
)
primary_fused = events.merge(
    primary_features,
    on="event_id",
    how="left",
    validate="one_to_one",
)

label_derived_columns = {
    "nf3_binary_label_raw",
    "nf3_binary_label",
    "nf3_attack_category_raw",
    "nf3_attack_category",
    "nf3_binary_label_available_flag",
    "nf3_attack_category_available_flag",
    "cyber_attack_window_label",
    "nf3_attack_flow_count",
    "nf3_benign_flow_count",
    "nf3_unknown_binary_label_count",
    "nf3_attack_flow_fraction",
    "nf3_distinct_attack_category_count",
    "nf3_attack_categories_present",
    "nf3_attack_category_counts_json",
}
assert not (label_derived_columns & set(primary_fused.columns)), (
    "Label-derived NF3 columns must not appear in the model feature bank."
)

assert len(primary_fused) == len(events) == EXPECTED_POINT_EVENT_ROWS
assert primary_fused["event_id"].is_unique
assert len(window_features) == len(events) * len(ALIGNMENT_CONFIG["windows"])
assert not window_features.duplicated(["event_id", "window_name"]).any()
assert not targets.duplicated(["event_id", "window_name"]).any()

print("Primary fused rows:", f"{len(primary_fused):,}")
print("Primary fused columns:", len(primary_fused.columns))
print(
    "Events with at least one primary-window flow:",
    require_int(
        primary_fused["nf3_flow_count"].gt(0).sum(),
        "primary events with flows",
    ),
)
print(
    "Events with zero primary-window flows:",
    require_int(
        primary_fused["nf3_flow_count"].eq(0).sum(),
        "primary events without flows",
    ),
)
display(
    primary_fused[
        [
            "event_id",
            "aircraft_event_time_utc",
            "aligned_event_time_utc",
            "alignment_mode",
            "replay_cycle_index",
            "nf3_flow_count",
            "nf3_packet_total_sum",
            "nf3_byte_total_sum",
            "nf3_nearest_flow_start_lag_s",
        ]
    ].head()
)


## 7. Save outputs, checksums, and build metadata

In [ ]:
# Export the feature, target, provenance, graph-edge, and metadata artifacts.
window_features.to_csv(WINDOW_FEATURES_OUT, index=False)
targets.to_csv(TARGETS_OUT, index=False)
primary_fused.to_csv(PRIMARY_FUSED_OUT, index=False)
flow_labels.to_csv(FLOW_LABELS_OUT, index=False)


def infer_role(table_name: str, column: str) -> str:
    if table_name in {"event_nf3_targets_v1", "nf3_flow_labels_v1"}:
        return "target_only"
    if column in {"event_id", "flow_id"}:
        return "identifier"
    if "time" in column or column.endswith("_utc"):
        return "temporal"
    if column.startswith("nf3_"):
        return "cyber_feature"
    if (
        column.startswith("alignment_")
        or column.startswith("window_")
        or column == "replay_cycle_index"
    ):
        return "alignment_metadata"
    return "physical_or_context_feature"


def describe_column(column: str) -> str:
    fixed = {
        "event_id": "Deterministic candidate point-event identifier from Notebook 01.",
        "flow_id": "Deterministic NF3 row identifier based on source row number.",
        "alignment_version": "Version of the point-event-to-NF3 alignment specification.",
        "alignment_mode": "Direct UTC or controlled cyclic replay mapping.",
        "alignment_is_controlled": "1 when the physical/cyber pairing is modeled rather than site-coincident.",
        "physical_event_time_utc": "Original aircraft-reported physical event timestamp.",
        "aligned_event_time_utc": "Timestamp used to query the NF3 replay timeline.",
        "alignment_time_shift_s": "Aligned timestamp minus original physical timestamp, in seconds.",
        "replay_cycle_index": "Zero-based cycle number for controlled cyclic replay.",
        "window_name": "Named temporal-window configuration.",
        "window_half_width_s": "Seconds before and after aligned event time.",
        "window_boundary_truncated_flag": "1 when a requested window was clipped at an NF3 replay boundary.",
        "cyber_attack_window_label": "Target-only indicator that one or more matched flows had binary NF3 Label=1.",
        "nf3_binary_label_raw": "Original flow-level value from the NF3 Label column.",
        "nf3_binary_label": "Normalized nullable integer flow-level binary label.",
        "nf3_attack_category_raw": "Original flow-level value from the NF3 Attack column.",
        "nf3_attack_category": "Trimmed flow-level NF3 attack category.",
        "nf3_attack_categories_present": "Pipe-delimited non-benign attack categories in the event window.",
        "nf3_attack_category_counts_json": "JSON object containing attack-category flow counts in the event window.",
    }
    if column in fixed:
        return fixed[column]
    if column.startswith("nf3_"):
        return "NF3-derived aggregate or relationship field; see the column name for statistic and unit."
    return "Inherited physical, episode, or backward-looking recurrence field from Notebook 02."


dictionary_rows: list[dict[str, str]] = []
for table_name, frame in [
    ("event_nf3_window_features_v1", window_features),
    ("event_nf3_targets_v1", targets),
    ("candidate_point_events_v1_with_context_and_nf3_primary", primary_fused),
    ("nf3_flow_labels_v1", flow_labels),
]:
    for column in frame.columns:
        dictionary_rows.append(
            {
                "table": table_name,
                "column": column,
                "dtype": str(frame[column].dtype),
                "role": infer_role(table_name, column),
                "description": describe_column(column),
            }
        )

if EDGES_OUT.exists():
    edge_header = pd.read_csv(EDGES_OUT, nrows=5)
    for column in edge_header.columns:
        dictionary_rows.append(
            {
                "table": "event_flow_edges_primary_v1",
                "column": column,
                "dtype": str(edge_header[column].dtype),
                "role": "graph_edge_or_relation",
                "description": describe_column(column),
            }
        )

data_dictionary = pd.DataFrame(dictionary_rows).drop_duplicates(["table", "column"])
data_dictionary.to_csv(DICTIONARY_OUT, index=False)

window_summary: dict[str, dict[str, int | float]] = {}
for window_name, group in window_features.groupby("window_name", sort=False):
    window_summary[str(window_name)] = {
        "rows": len(group),
        "events_with_flows": require_int(
            group["nf3_flow_count"].gt(0).sum(),
            f"{window_name} events_with_flows",
        ),
        "events_without_flows": require_int(
            group["nf3_flow_count"].eq(0).sum(),
            f"{window_name} events_without_flows",
        ),
        "total_matched_flow_relationships": require_int(
            group["nf3_flow_count"].sum(),
            f"{window_name} total_matched_flow_relationships",
        ),
        "median_flow_count": float(group["nf3_flow_count"].median()),
        "maximum_flow_count": require_int(
            group["nf3_flow_count"].max(),
            f"{window_name} maximum_flow_count",
        ),
        "boundary_truncated_events": require_int(
            group["window_boundary_truncated_flag"].sum(),
            f"{window_name} boundary_truncated_events",
        ),
    }

primary_targets = targets.loc[targets["window_name"].eq(primary_window_name)]
target_summary = {
    "binary_label_column_available": True,
    "attack_category_column_available": True,
    "primary_positive_windows": require_int(
        primary_targets["cyber_attack_window_label"].fillna(0).sum(),
        "primary_positive_windows",
    ),
    "primary_negative_windows": require_int(
        primary_targets["cyber_attack_window_label"].eq(0).sum(),
        "primary_negative_windows",
    ),
    "primary_distinct_attack_categories": sorted(
        {
            category
            for value in primary_targets["nf3_attack_categories_present"].dropna()
            for category in str(value).split("|")
            if category
        }
    ),
}

summary = {
    "definitions": ALIGNMENT_CONFIG,
    "interpretation": {
        "site_coincident_claim": False,
        "causal_claim": False,
        "controlled_mapping_active": bool(alignment_mode != "direct_utc"),
        "statement": (
            "Physical/cyber relationships are controlled temporal pairings unless direct_utc "
            "is explicitly used with genuinely overlapping site-coincident data."
        ),
    },
    "inputs": {
        "context_file": str(CONTEXT_INPUT.relative_to(PROJECT_ROOT)),
        "context_rows": len(events),
        "netflow_file": str(NETFLOW_FILE.relative_to(PROJECT_ROOT)),
        "netflow_source_rows": source_netflow_rows,
        "netflow_valid_interval_rows": len(net),
        "netflow_invalid_interval_rows": invalid_netflow_time_rows,
        "netflow_time_start_utc": nf3_start.isoformat(),
        "netflow_time_end_utc": nf3_end.isoformat(),
        "resolved_schema": {
            **resolved,
            "packet_columns": packet_columns,
            "byte_columns": byte_columns,
            "iat_columns": iat_columns,
        },
    },
    "alignment": {
        "requested_mode": requested_mode,
        "selected_mode": alignment_mode,
        "direct_utc_coverage_fraction": direct_coverage_fraction,
        "replay_cycles_used": require_int(
            mapped["replay_cycle_index"].max(),
            "maximum replay_cycle_index",
        ) + 1,
    },
    "outputs": {
        "primary_fused_rows": len(primary_fused),
        "primary_fused_columns": len(primary_fused.columns),
        "window_feature_rows": len(window_features),
        "target_rows": len(targets),
        "flow_label_rows": len(flow_labels),
        "primary_edge_rows": primary_edge_count,
        "windows": window_summary,
        "targets": target_summary,
    },
    "regression_checks": [
        {
            "metric": "input_point_events",
            "expected": EXPECTED_POINT_EVENT_ROWS,
            "actual": len(events),
            "match": len(events) == EXPECTED_POINT_EVENT_ROWS,
        },
        {
            "metric": "primary_fused_rows",
            "expected": EXPECTED_POINT_EVENT_ROWS,
            "actual": len(primary_fused),
            "match": len(primary_fused) == EXPECTED_POINT_EVENT_ROWS,
        },
        {
            "metric": "window_rows",
            "expected": EXPECTED_POINT_EVENT_ROWS * len(ALIGNMENT_CONFIG["windows"]),
            "actual": len(window_features),
            "match": (
                len(window_features)
                == EXPECTED_POINT_EVENT_ROWS * len(ALIGNMENT_CONFIG["windows"])
            ),
        },
        {
            "metric": "flow_label_rows",
            "expected": len(net),
            "actual": len(flow_labels),
            "match": len(flow_labels) == len(net),
        },
        {
            "metric": "label_leakage_columns_in_primary_feature_bank",
            "expected": 0,
            "actual": len(label_derived_columns & set(primary_fused.columns)),
            "match": len(label_derived_columns & set(primary_fused.columns)) == 0,
        },
    ],
}

if ALIGNMENT_CONFIG["compute_sha256"]:
    summary["input_sha256"] = {
        "context_file": sha256_file(CONTEXT_INPUT),
        "netflow_file": sha256_file(NETFLOW_FILE),
    }

with SUMMARY_OUT.open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2, default=str)

artifact_paths = [
    WINDOW_FEATURES_OUT,
    TARGETS_OUT,
    PRIMARY_FUSED_OUT,
    EDGES_OUT,
    FLOW_LABELS_OUT,
    SUMMARY_OUT,
    DICTIONARY_OUT,
]
checksum_lines: list[str] = []
for path in artifact_paths:
    if path.exists():
        checksum_lines.append(f"{sha256_file(path)}  {path.name}")
CHECKSUMS_OUT.write_text("\n".join(checksum_lines) + "\n", encoding="utf-8")

print("Saved outputs:")
for path in [*artifact_paths, CHECKSUMS_OUT]:
    if path.exists():
        print(f"  {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size:,} bytes)")


## 8. Final validation summary

In [ ]:
assert all(check["match"] for check in summary["regression_checks"]), summary["regression_checks"]

validation_table = pd.DataFrame(summary["regression_checks"])
display(validation_table)

display(
    pd.DataFrame(
        [
            {"metric": "Alignment mode", "value": alignment_mode},
            {
                "metric": "Controlled mapping active",
                "value": alignment_mode != "direct_utc",
            },
            {"metric": "Input point events", "value": len(events)},
            {"metric": "Primary fused rows", "value": len(primary_fused)},
            {"metric": "Sensitivity rows", "value": len(window_features)},
            {"metric": "Primary event-flow edges", "value": primary_edge_count},
            {
                "metric": "Primary events with flows",
                "value": require_int(
                    primary_fused["nf3_flow_count"].gt(0).sum(),
                    "primary events with flows",
                ),
            },
            {
                "metric": "Primary events without flows",
                "value": require_int(
                    primary_fused["nf3_flow_count"].eq(0).sum(),
                    "primary events without flows",
                ),
            },
            {
                "metric": "NF3 binary labels available",
                "value": resolved["binary_label"] is not None,
            },
            {
                "metric": "NF3 attack categories available",
                "value": resolved["attack_category"] is not None,
            },
            {"metric": "Flow-label provenance rows", "value": len(flow_labels)},
        ]
    )
)

print("Notebook 03 completed successfully.")
print("The next stage is 04_point_event_tabular_baselines.ipynb.")
